# 01. Attention Variants: MHA, MQA, GQA, MLA, and DiffAttn

- **Category:** 05_architectures / 04_transformers_and_heads
- **Lineage & Citations:** Ashish Vaswani et al., 2017 (*Attention Is All You Need*); Noam Shazeer, 2019 (*MQA*); Joshua Ainslie et al., 2023 (*GQA*); DeepSeek AI, 2024 (*DeepSeek-V2 MLA*); Tianzhe Ye et al., 2024 (*Differential Transformer*).
- **Purpose:** Explores architectural strategies to conquer the KV cache memory wall and eliminate attention noise—from single-head KV sharing (MQA) and grouped partitioning (GQA) to low-rank latent compression (MLA) and noise cancellation (DiffAttn).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Reservoir Computing Echo State Networks LSM and NG-RC](../03_recurrent_ssm_and_memory/07_reservoir_computing_echo_state_networks_lsm_and_ngrc.ipynb) $\to$ `🔗 Downstream Syntheses:` [Transformer Heads Induction OV QK Circuits MTP and Registers](02_transformer_heads_induction_ov_qk_circuits_mtp_and_registers.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> - **MQA on 70B+ Reasoning Models:** Pure MQA ($G = 1$) can cause degradation on multi-hop logical reasoning; prefer GQA ($G = 8$).
> - **DiffAttn on Short Sequences (< 64 tokens):** Added projection parameter overhead outweighs noise-cancelling benefits on short prompts.

## Mathematical Foundations & Governing Equations

### 1. Grouped-Query Attention (GQA)
Partitioning $H$ query heads into $G$ groups ($H / G$ heads per KV group):
$$
g = \lfloor h / (H/G) \rfloor, \quad \text{scores}_{h} = \frac{q_h k_g^\top}{\sqrt{D}}, \quad A_h = \text{softmax}(\text{scores}_h), \quad O_h = A_h v_g
$$

### 2. Differential Attention (DiffAttn)
Subtractive attention map cancellation:
$$
A_{\text{diff}} = \text{softmax}\left(\frac{Q_1 K_1^\top}{\sqrt{D}}\right) - \lambda \cdot \text{softmax}\left(\frac{Q_2 K_2^\top}{\sqrt{D}}\right), \quad O = A_{\text{diff}} V
$$

In [ ]:
import math
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/05_architectures/04_transformers_and_heads/01_attention_variants_mha_mqa_gqa_mla_and_diffattn.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)

q, k, v = weights["q"], weights["k"], weights["v"]
scale = float(weights["scale"][0])
lambda_val = float(weights["lambda_val"][0])
print(f"Loaded {len(weights)} golden tensors on {env.device}, Q shape: {list(q.shape)}, KV shape: {list(k.shape)}")

In [ ]:
def grouped_query_attention(q, k, v, scale):
    B, H, T, D = q.shape
    G, S = k.shape[1], k.shape[2]
    group_size = H // G
    k_exp = k.unsqueeze(2).expand(B, G, group_size, S, D).reshape(B, H, S, D)
    v_exp = v.unsqueeze(2).expand(B, G, group_size, S, D).reshape(B, H, S, D)
    scores = torch.matmul(q, k_exp.transpose(-1, -2)) * scale
    attn_weights = F.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, v_exp), attn_weights

def differential_attention(q1, k1, q2, k2, v, lambda_param, scale):
    a1 = F.softmax(torch.matmul(q1, k1.transpose(-1, -2)) * scale, dim=-1)
    a2 = F.softmax(torch.matmul(q2, k2.transpose(-1, -2)) * scale, dim=-1)
    diff_weights = a1 - lambda_param * a2
    return torch.matmul(diff_weights, v), diff_weights

In [ ]:
# Quantitative Mathematical Assertions & Golden Parity Verification
gqa_out, gqa_weights = grouped_query_attention(q, k, v, scale)
err_g = float(torch.max(torch.abs(gqa_out - weights["gqa_out"])))
err_gw = float(torch.max(torch.abs(gqa_weights - weights["gqa_weights"])))
assert err_g < 1e-6, f"GQA output error: {err_g:.2e}"
assert err_gw < 1e-6, f"GQA weights error: {err_gw:.2e}"

diff_out, diff_weights = differential_attention(
    weights["q1"], weights["k1"], weights["q2"], weights["k2"],
    weights["v_diff"], lambda_val, scale
)
err_d = float(torch.max(torch.abs(diff_out - weights["diff_out"])))
err_dw = float(torch.max(torch.abs(diff_weights - weights["diff_weights"])))
assert err_d < 1e-6, f"DiffAttn output error: {err_d:.2e}"
assert err_dw < 1e-6, f"DiffAttn weights error: {err_dw:.2e}"
print(f"Parity Verified: GQA={err_g:.2e}, DiffAttn={err_d:.2e}")

In [ ]:
import matplotlib.pyplot as plt
from core.benchmark import benchmark_fn, format_benchmark_table

b_gqa = benchmark_fn(lambda: grouped_query_attention(q, k, v, scale), iters=50, name="Grouped-Query Attention")
b_diff = benchmark_fn(lambda: differential_attention(
    weights["q1"], weights["k1"], weights["q2"], weights["k2"],
    weights["v_diff"], lambda_val, scale
), iters=50, name="Differential Attention")
print(format_benchmark_table([b_gqa, b_diff]))

# Visualization: GQA Attention Map & Differential Attention Noise-Cancelled Map
fig, ax = plt.subplots(1, 2, figsize=(8, 2.5), dpi=100)
ax[0].imshow(gqa_weights[0, 0].detach().numpy(), cmap="viridis", aspect="auto")
ax[0].set_title("GQA Attention Weights (Head 0)")
ax[1].imshow(diff_weights[0, 0].detach().numpy(), cmap="coolwarm", aspect="auto")
ax[1].set_title("DiffAttn Weights (A1 - λ A2)")
plt.tight_layout()
plt.show()

In [ ]:
# Pathological Misuse: Over-Cancellation with lambda >= 1.0 in Differential Attention
# When lambda >= 1.0, row sums become <= 0, causing negative attention weights and inverted activations!
lambda_pathological = 1.2
_, diff_bad = differential_attention(
    weights["q1"], weights["k1"], weights["q2"], weights["k2"],
    weights["v_diff"], lambda_pathological, scale
)
row_sums = diff_bad.sum(dim=-1)
print(f"Pathological row sums with lambda=1.2: {row_sums[0, 0, 0].item():.2f}")
assert (row_sums < 0.0).any(), "lambda > 1.0 must produce negative net attention weight!"
print("Demonstrated requirement of lambda < 1.0 for valid Differential Attention normalization.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **GQA is the Modern LLM Default:** Striking the optimal Pareto frontier between MHA quality and MQA memory efficiency.
2. **MLA Slashes Serving Memory:** DeepSeek's latent compression reduces KV cache storage by up to $93\%$.
3. **DiffAttn Sharpens Salience:** Subtractive attention cancels non-informative prompt hallucinations.

---
### Cross-Domain Generalisation Matrix
- **High-Concurrency LLM Inference:** GQA unlocks $4\times - 8\times$ larger KV cache batch sizes per GPU.
- **Needle Retrieval in Long Documents:** Differential attention filters distractor tokens in massive prompts.
- **Mobile Unified Memory:** MLA enables 16-head attention models to run within smartphone memory budgets.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_attention_variants_mha_mqa_gqa_mla_and_diffattn.md](../../../markdown/05_architectures/04_transformers_and_heads/01_attention_variants_mha_mqa_gqa_mla_and_diffattn.md)
- **PyTorch 2.x Module:** [01_attention_variants_mha_mqa_gqa_mla_and_diffattn.py](../../../pytorch2/05_architectures/04_transformers_and_heads/01_attention_variants_mha_mqa_gqa_mla_and_diffattn.py)
- **Functional JAX Module:** [01_attention_variants_mha_mqa_gqa_mla_and_diffattn.py](../../../jax/05_architectures/04_transformers_and_heads/01_attention_variants_mha_mqa_gqa_mla_and_diffattn.py)
- **Candle Rust Module:** [01_attention_variants_mha_mqa_gqa_mla_and_diffattn.rs](../../../candle/05_architectures/04_transformers_and_heads/01_attention_variants_mha_mqa_gqa_mla_and_diffattn.rs)
- **Hardware Kernel:** [01_attention_variants_mha_mqa_gqa_mla_and_diffattn.rs](../../../kernels/05_architectures/04_transformers_and_heads/01_attention_variants_mha_mqa_gqa_mla_and_diffattn.rs)
- **Universal Golden SafeTensors:** [01_attention_variants_mha_mqa_gqa_mla_and_diffattn.safetensors](../../../weights/05_architectures/04_transformers_and_heads/01_attention_variants_mha_mqa_gqa_mla_and_diffattn.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous Topic: 07. Reservoir Computing Echo State Networks LSM and NG-RC](../03_recurrent_ssm_and_memory/07_reservoir_computing_echo_state_networks_lsm_and_ngrc.ipynb) | [Category Index](../../../README.md#05_architectures) | [Next Topic: 02. Transformer Heads Induction OV QK Circuits MTP and Registers](02_transformer_heads_induction_ov_qk_circuits_mtp_and_registers.ipynb) &rarr;